# Sanskrit-to-English Neural Machine Translation
### Assignment 2 — Natural Language Understanding

A simple sequence-to-sequence (seq2seq) translation model, built from scratch in PyTorch:

**Encoder** (reads Sanskrit) → **Attention** (looks back at the encoder while translating) →
**Decoder** (writes English), with **beam search** used to produce the final translations.

No pretrained translation model is used. The only pretrained model anywhere in this notebook is
the BERT model *inside* the `bert-score` library — that's just used to compute the BERTScore
evaluation metric (as the assignment requires), never for translation itself.

**To run this notebook:** just run all cells top to bottom — the dataset (`acomquest/Saamayik`
on HuggingFace) is downloaded automatically in Section 3, no manual file setup needed.


## 1. Install & Import Libraries

In [ ]:
!pip install torch sentencepiece nltk bert-score pandas matplotlib datasets --quiet

import os, re, time, random, json
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import sentencepiece as spm
from datasets import load_dataset
from nltk.translate.bleu_score import corpus_bleu
from bert_score import score as bert_score

random.seed(0)
torch.manual_seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
print(torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## 2. Settings
All the numbers we might want to tune live here, in one place.

In [ ]:
OUT_DIR = "new_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

VOCAB_SIZE = 8000      # subword vocabulary size (same for Sanskrit and English)
MAX_LEN = 60           # sentences longer than this get cut short

EMBED_SIZE = 256
HIDDEN_SIZE = 512
DROPOUT = 0.3

BATCH_SIZE = 64
EPOCHS = 30
LEARNING_RATE = 0.001

BEAM_WIDTH = 5          # used only when translating (not during training)

PAD, SOS, EOS, UNK = 0, 1, 2, 3   # special token ids, fixed for both languages

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# EDIT this path to wherever your 6 CSVs live
DATA_DIR = '/content/drive/MyDrive/data imp_sanskrit/'

## 3. Load the Data
The dataset is loaded directly from HuggingFace: `acomquest/Saamayik`. Each example has a `translation` field, a dict with `en` and `sn` keys, plus a `source` field naming which sub-corpus it came from (Bible, NIOS, Spoken Tutorials, etc.) — we don't need `source` for translation itself, just `translation['en']` and `translation['sn']`.

The HuggingFace split names don't exactly match the assignment's (`validation` here = `dev` in the assignment), so we rename them. We also add our own `Source_id` (just the row number within each split) since the raw dataset doesn't have one.

In [ ]:

def load_split(split):
    suffix = '_10000' if split == 'train' else '_1000'
    sa = pd.read_csv(f'{DATA_DIR}{split}_sa{suffix}.csv')
    en = pd.read_csv(f'{DATA_DIR}{split}_en{suffix}.csv')
    df = pd.merge(sa, en, on='Source_id')
    return df

train_df = load_split('train')
dev_df   = load_split('dev')
test_df  = load_split('test')

print('train:', train_df.shape, ' dev:', dev_df.shape, ' test:', test_df.shape)
train_df.head()

## 4. Tokenization
Sanskrit words change form a lot (declensions, sandhi, long compound words), so splitting by whitespace alone would create a huge vocabulary with lots of rare/unseen words. Instead we train a **subword tokenizer** (SentencePiece, BPE) that breaks words into smaller, reusable pieces. We train it ourselves on the training data only — it's not a pretrained model.

In [ ]:
# SentencePiece needs plain text files to train on, one sentence per line.
train_df["Sentence_sa"].to_csv(f"{OUT_DIR}/sa.txt", index=False, header=False)
train_df["Sentence_en"].to_csv(f"{OUT_DIR}/en.txt", index=False, header=False)

spm.SentencePieceTrainer.train(
    input=f"{OUT_DIR}/sa.txt", model_prefix=f"{OUT_DIR}/sa", vocab_size=VOCAB_SIZE,
    pad_id=PAD, bos_id=SOS, eos_id=EOS, unk_id=UNK, character_coverage=1.0,
)
spm.SentencePieceTrainer.train(
    input=f"{OUT_DIR}/en.txt", model_prefix=f"{OUT_DIR}/en", vocab_size=VOCAB_SIZE,
    pad_id=PAD, bos_id=SOS, eos_id=EOS, unk_id=UNK, character_coverage=1.0,
)

sp_sa = spm.SentencePieceProcessor(model_file=f"{OUT_DIR}/sa.model")
sp_en = spm.SentencePieceProcessor(model_file=f"{OUT_DIR}/en.model")

def encode(text, sp):
    # turn a sentence into a list of token ids, with <sos> ... <eos> wrapped around it
    ids = sp.encode(text, out_type=int)[:MAX_LEN - 2]
    return [SOS] + ids + [EOS]

def decode(ids, sp):
    # turn ids back into a sentence, dropping special tokens
    ids = [i for i in ids if i not in (PAD, SOS, EOS)]
    return sp.decode(ids)

print("Sanskrit vocab size:", sp_sa.get_piece_size())
print("English vocab size:", sp_en.get_piece_size())
print("Example:", encode(train_df['Sentence_sa'][0], sp_sa))


## 5. PyTorch Dataset
Turns each sentence pair into token ids, and pads each batch so all sentences in it have the same length.

In [ ]:
class TranslationDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        self.src = [encode(s, sp_sa) for s in df["Sentence_sa"]]
        self.tgt = [encode(s, sp_en) for s in df["Sentence_en"]]

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        return self.src[idx], self.tgt[idx]

def collate_fn(batch):
    src_list, tgt_list = zip(*batch)
    src_max = max(len(s) for s in src_list)
    tgt_max = max(len(t) for t in tgt_list)

    src_batch = torch.full((len(batch), src_max), PAD, dtype=torch.long)
    tgt_batch = torch.full((len(batch), tgt_max), PAD, dtype=torch.long)
    for i, (s, t) in enumerate(zip(src_list, tgt_list)):
        src_batch[i, :len(s)] = torch.tensor(s)
        tgt_batch[i, :len(t)] = torch.tensor(t)
    return src_batch, tgt_batch

train_loader = torch.utils.data.DataLoader(TranslationDataset(train_df), batch_size=BATCH_SIZE,
                                            shuffle=True, collate_fn=collate_fn)
dev_loader = torch.utils.data.DataLoader(TranslationDataset(dev_df), batch_size=BATCH_SIZE,
                                          shuffle=False, collate_fn=collate_fn)
print("train batches:", len(train_loader))


## 6. The Model
Three small building blocks, chained together:

- **Encoder** — reads the Sanskrit sentence with an LSTM and produces one vector per token.
- **Attention** — at every step of decoding, looks at all encoder vectors and decides which ones matter most right now (a weighted average).
- **Decoder** — an LSTM that, at each step, takes the previous English word plus the attention output, and predicts the next English word.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, EMBED_SIZE, padding_idx=PAD)
        self.lstm = nn.LSTM(EMBED_SIZE, HIDDEN_SIZE, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(DROPOUT)
        # the encoder is bidirectional (reads left-to-right AND right-to-left), so its output
        # is twice as wide as HIDDEN_SIZE. We shrink it back down to HIDDEN_SIZE here.
        self.reduce_h = nn.Linear(HIDDEN_SIZE * 2, HIDDEN_SIZE)
        self.reduce_c = nn.Linear(HIDDEN_SIZE * 2, HIDDEN_SIZE)

    def forward(self, src):
        embedded = self.dropout(self.embedding(src))          # (batch, seq_len, embed)
        outputs, (h, c) = self.lstm(embedded)                  # outputs: (batch, seq_len, 2*hidden)
        # h and c each have shape (2, batch, hidden) -- one for forward, one for backward.
        # Concatenate them and project down to a single (batch, hidden) starting state.
        h = torch.tanh(self.reduce_h(torch.cat([h[0], h[1]], dim=1)))
        c = torch.tanh(self.reduce_c(torch.cat([c[0], c[1]], dim=1)))
        return outputs, h, c


class Attention(nn.Module):
    # Bahdanau-style additive attention.
    def __init__(self):
        super().__init__()
        self.attn = nn.Linear(HIDDEN_SIZE * 2 + HIDDEN_SIZE, HIDDEN_SIZE)
        self.v = nn.Linear(HIDDEN_SIZE, 1, bias=False)

    def forward(self, decoder_hidden, encoder_outputs, src_mask):
        seq_len = encoder_outputs.size(1)
        # repeat the decoder's current hidden state so we can compare it against every
        # encoder position at once
        hidden_rep = decoder_hidden.unsqueeze(1).repeat(1, seq_len, 1)
        energy = torch.tanh(self.attn(torch.cat([hidden_rep, encoder_outputs], dim=2)))
        scores = self.v(energy).squeeze(2)                     # (batch, seq_len)
        scores = scores.masked_fill(src_mask == 0, float("-inf"))
        weights = F.softmax(scores, dim=1)                     # how much to attend to each position
        context = torch.bmm(weights.unsqueeze(1), encoder_outputs).squeeze(1)  # weighted sum
        return context, weights


class Decoder(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, EMBED_SIZE, padding_idx=PAD)
        self.attention = Attention()
        self.lstm = nn.LSTM(EMBED_SIZE + HIDDEN_SIZE * 2, HIDDEN_SIZE, batch_first=True)
        self.fc_out = nn.Linear(HIDDEN_SIZE * 3 + EMBED_SIZE, vocab_size)
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, input_token, h, c, encoder_outputs, src_mask):
        embedded = self.dropout(self.embedding(input_token)).unsqueeze(1)   # (batch, 1, embed)
        context, weights = self.attention(h, encoder_outputs, src_mask)
        lstm_input = torch.cat([embedded, context.unsqueeze(1)], dim=2)
        output, (h, c) = self.lstm(lstm_input, (h.unsqueeze(0), c.unsqueeze(0)))
        h, c = h.squeeze(0), c.squeeze(0)
        output = output.squeeze(1)
        prediction = self.fc_out(torch.cat([output, context, embedded.squeeze(1)], dim=1))
        return prediction, h, c


class Seq2Seq(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size):
        super().__init__()
        self.encoder = Encoder(src_vocab_size)
        self.decoder = Decoder(tgt_vocab_size)

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        batch_size, tgt_len = tgt.shape
        vocab_size = self.decoder.fc_out.out_features
        outputs = torch.zeros(batch_size, tgt_len, vocab_size).to(src.device)

        src_mask = (src != PAD).long()
        encoder_outputs, h, c = self.encoder(src)

        input_token = tgt[:, 0]  # first token is always <sos>
        for t in range(1, tgt_len):
            prediction, h, c = self.decoder(input_token, h, c, encoder_outputs, src_mask)
            outputs[:, t] = prediction
            use_teacher_forcing = random.random() < teacher_forcing_ratio
            input_token = tgt[:, t] if use_teacher_forcing else prediction.argmax(1)
        return outputs


model = Seq2Seq(sp_sa.get_piece_size(), sp_en.get_piece_size()).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {n_params:,}")


## 7. Training
Standard supervised training: predict the next English token at every position, compare to the real one with cross-entropy loss, backpropagate.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss(ignore_index=PAD)

def train_one_epoch():
    model.train()
    total_loss = 0
    for src, tgt in train_loader:
        src, tgt = src.to(device), tgt.to(device)
        optimizer.zero_grad()
        output = model(src, tgt)                 # (batch, tgt_len, vocab)
        output = output[:, 1:].reshape(-1, output.shape[-1])
        target = tgt[:, 1:].reshape(-1)
        loss = criterion(output, target)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # avoid exploding gradients
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)

@torch.no_grad()
def evaluate_loss(loader):
    model.eval()
    total_loss = 0
    for src, tgt in loader:
        src, tgt = src.to(device), tgt.to(device)
        output = model(src, tgt, teacher_forcing_ratio=0.0)
        output = output[:, 1:].reshape(-1, output.shape[-1])
        target = tgt[:, 1:].reshape(-1)
        total_loss += criterion(output, target).item()
    return total_loss / len(loader)

@torch.no_grad()
def translate_greedy(sentence):
    # Fast, simple decoding used only to track BLEU during training (not for final results).
    model.eval()
    src = torch.tensor([encode(sentence, sp_sa)]).to(device)
    src_mask = (src != PAD).long()
    encoder_outputs, h, c = model.encoder(src)

    input_token = torch.tensor([SOS]).to(device)
    output_ids = []
    for _ in range(MAX_LEN):
        prediction, h, c = model.decoder(input_token, h, c, encoder_outputs, src_mask)
        next_id = prediction.argmax(1).item()
        if next_id == EOS:
            break
        output_ids.append(next_id)
        input_token = torch.tensor([next_id]).to(device)
    return decode(output_ids, sp_en)

def bleu_score(predictions, references):
    refs = [[r.split()] for r in references]
    hyps = [p.split() for p in predictions]
    return corpus_bleu(refs, hyps)


In [ ]:
train_losses, dev_losses, dev_bleus = [], [], []
best_bleu = -1

for epoch in range(1, EPOCHS + 1):
    start = time.time()
    train_loss = train_one_epoch()
    dev_loss = evaluate_loss(dev_loader)

    dev_predictions = [translate_greedy(s) for s in dev_df["Sentence_sa"]]
    dev_bleu = bleu_score(dev_predictions, dev_df["Sentence_en"].tolist())

    train_losses.append(train_loss)
    dev_losses.append(dev_loss)
    dev_bleus.append(dev_bleu)

    print(f"Epoch {epoch:2d} | train loss {train_loss:.3f} | dev loss {dev_loss:.3f} "
          f"| dev BLEU {dev_bleu:.4f} | {time.time()-start:.1f}s")

    if dev_bleu > best_bleu:
        best_bleu = dev_bleu
        torch.save(model.state_dict(), f"{OUT_DIR}/best_model.pt")

model.load_state_dict(torch.load(f"{OUT_DIR}/best_model.pt"))
print(f"\nBest dev BLEU: {best_bleu:.4f} -- loaded best checkpoint.")


### Save a Full Checkpoint (so you never have to retrain again)

In [ ]:
checkpoint = {
    "model_state_dict": model.state_dict(),
    "embed_size": EMBED_SIZE,
    "hidden_size": HIDDEN_SIZE,
    "dropout": DROPOUT,
    "src_vocab_size": sp_sa.get_piece_size(),
    "tgt_vocab_size": sp_en.get_piece_size(),
}
torch.save(checkpoint, f"{OUT_DIR}/checkpoint.pt")
print(f"Saved full checkpoint to {OUT_DIR}/checkpoint.pt")
print("(The SentencePiece tokenizer files outputs/sa.model and outputs/en.model, saved back "
      "in Section 4, are also needed to reload -- keep them alongside checkpoint.pt.)")


### Reload the Model Later (no retraining needed)

Next time you open this notebook, you don't need to rerun the training loop at all. Just run,
in order:

1. **Section 1** (installs + imports)
2. **Section 2** (settings)
3. **Section 6** (the `Encoder` / `Attention` / `Decoder` / `Seq2Seq` class definitions —
   just the class definitions, not the line that creates a fresh `model`)
4. The cell directly below this one

That's it — skip Sections 3, 4 (data loading/tokenizer *training*), 5, and 7 entirely. This cell
loads the saved tokenizers and the saved weights instead of retraining anything.


In [ ]:
# Load the tokenizers (trained once, saved to disk -- no need to retrain them)
sp_sa = spm.SentencePieceProcessor(model_file=f"{OUT_DIR}/sa.model")
sp_en = spm.SentencePieceProcessor(model_file=f"{OUT_DIR}/en.model")

def encode(text, sp):
    ids = sp.encode(text, out_type=int)[:MAX_LEN - 2]
    return [SOS] + ids + [EOS]

def decode(ids, sp):
    ids = [i for i in ids if i not in (PAD, SOS, EOS)]
    return sp.decode(ids)

# Rebuild the model with the same architecture settings it was trained with, then load weights
checkpoint = torch.load(f"{OUT_DIR}/checkpoint.pt", map_location=device)
model = Seq2Seq(checkpoint["src_vocab_size"], checkpoint["tgt_vocab_size"]).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Model loaded from checkpoint -- ready to translate, no training required.")
print(f"(trained with embed_size={checkpoint['embed_size']}, hidden_size={checkpoint['hidden_size']})")


### Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(train_losses, label="train loss")
axes[0].plot(dev_losses, label="dev loss")
axes[0].set_xlabel("epoch"); axes[0].legend(); axes[0].set_title("Loss")

axes[1].plot(dev_bleus, color="green")
axes[1].set_xlabel("epoch"); axes[1].set_title("Dev BLEU")

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/training_curves.png", dpi=150)
plt.show()


## 8. Beam Search
Greedy decoding (used above) always picks the single most likely next word, which can lock in an early mistake. Beam search instead keeps track of the `BEAM_WIDTH` most promising partial translations at every step, so it can recover from an early low-probability choice if the rest of the sentence turns out better.

In [ ]:
@torch.no_grad()
def translate_beam(sentence, beam_width=BEAM_WIDTH):
    model.eval()
    src = torch.tensor([encode(sentence, sp_sa)]).to(device)
    src_mask = (src != PAD).long()
    encoder_outputs, h, c = model.encoder(src)

    # each beam is: (list_of_token_ids, hidden_state, cell_state, log_probability, finished?)
    beams = [([SOS], h, c, 0.0, False)]

    for _ in range(MAX_LEN):
        candidates = []
        for tokens, h_i, c_i, log_prob, finished in beams:
            if finished:
                candidates.append((tokens, h_i, c_i, log_prob, True))
                continue
            input_token = torch.tensor([tokens[-1]]).to(device)
            prediction, h_new, c_new = model.decoder(input_token, h_i, c_i, encoder_outputs, src_mask)
            log_probs = F.log_softmax(prediction, dim=1).squeeze(0)
            top_log_probs, top_ids = log_probs.topk(beam_width)
            for lp, idx in zip(top_log_probs.tolist(), top_ids.tolist()):
                is_finished = (idx == EOS)
                candidates.append((tokens + [idx], h_new, c_new, log_prob + lp, is_finished))

        # keep only the best `beam_width` candidates, ranked by total log-probability
        candidates.sort(key=lambda x: x[3], reverse=True)
        beams = candidates[:beam_width]

        if all(finished for _, _, _, _, finished in beams):
            break

    best_tokens = beams[0][0]
    return decode(best_tokens, sp_en)


## 9. Final Evaluation — BLEU, BERTScore, Efficiency
Final numbers use **beam search** (not the fast greedy decoding used during training).

In [ ]:
def translate_all(df, beam_width=BEAM_WIDTH):
    return [translate_beam(s, beam_width) for s in df["Sentence_sa"]]

# ---- Dev set ----
dev_preds = translate_all(dev_df)
dev_bleu = bleu_score(dev_preds, dev_df["Sentence_en"].tolist())
_, _, dev_f1 = bert_score(dev_preds, dev_df["Sentence_en"].tolist(), lang="en", rescale_with_baseline=True)
dev_bertscore = dev_f1.mean().item()
print(f"DEV  -> BLEU: {dev_bleu:.4f} | BERTScore-F1: {dev_bertscore:.4f}")

# ---- Test set (timed, for the efficiency metric) ----
start = time.time()
test_preds = translate_all(test_df)
inference_time = time.time() - start

test_bleu = bleu_score(test_preds, test_df["Sentence_en"].tolist())
_, _, test_f1 = bert_score(test_preds, test_df["Sentence_en"].tolist(), lang="en", rescale_with_baseline=True)
test_bertscore = test_f1.mean().item()

print(f"TEST -> BLEU: {test_bleu:.4f} | BERTScore-F1: {test_bertscore:.4f}")
print(f"Inference time on {len(test_df)} test sentences: {inference_time:.2f} sec")
print(f"Total trainable parameters: {n_params:,}")

results = {
    "dev_bleu": dev_bleu, "dev_bertscore_f1": dev_bertscore,
    "test_bleu": test_bleu, "test_bertscore_f1": test_bertscore,
    "test_inference_time_sec": inference_time, "total_parameters": n_params,
}
with open(f"{OUT_DIR}/results.json", "w") as f:
    json.dump(results, f, indent=2)


## 10. `submission.csv`

In [ ]:
submission = pd.DataFrame({
    "Source_id": test_df["Source_id"],
    "Sentence_en": test_preds,
})
submission.to_csv(f"{OUT_DIR}/submission.csv", index=False, encoding="utf-8")
print(f"Saved {len(submission)} predictions to {OUT_DIR}/submission.csv")
submission.head()


## 11. Translation Examples
A few examples to look at and discuss in the report.

In [ ]:
for i in random.sample(range(len(dev_df)), 8):
    print("SRC :", dev_df.iloc[i]["Sentence_sa"])
    print("REF :", dev_df.iloc[i]["Sentence_en"])
    print("PRED:", dev_preds[i])
    print("-" * 70)
